# 📊 Medical VLM Decisive Validation — Standalone Zero-GPU Aggregator (rev05 & rev06)

**Research Project:** Selective Answering for Bilingual Medical Vision-Language Models Under Language Shift (IndoRad-VQA)
**Execution Mode:** 100% CPU-Only (Zero GPU Memory Required)
**Revision:** `rev05` & `rev06` Unified Production Evaluator

---

### 📋 Scientific Scope of this Standalone Evaluator:
1. **Zero-GPU Efficiency:** Does not load multi-billion parameter neural networks. Evaluates finalized checkpoints directly on CPU in under 30 seconds.
2. **Dual-Model Statistical Rigor:**
   - Evaluates `Qwen/Qwen2.5-VL-7B-Instruct` and `FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL` (1,133 QA pairs each).
3. **RQ1 Bilingual Evaluation:** Cross-lingual accuracy (EN vs ID), prediction flip rate, Wilson 95% CI, and paired McNemar test.
4. **RQ2 Multimodal Visual Reliance:** Directional sensitivity analysis $\Delta P_{\text{dir}} = P(\hat{y} \mid P_0) - P(\hat{y} \mid P_1)$ and Visual Flip Rate (P0 vs P1).
5. **RQ3 Selective Prediction (Discrete Riemann AURC):**
   - Tie-aware random expectation AURC (default for rev06) and stable mergesort AURC (rev05 legacy).
   - Vectorized cluster-level paired permutation tests ($B=10,000$) with step-down Holm-Bonferroni family correction.
   - Cluster-level paired bootstrap ($B=1000$, seed 42) for 95% Confidence Intervals.
6. **RQ4 Learned Risk Modeling & Adaptive Abstention:**
   - Standardized multivariate logistic regression (`LR-full`, `LR-visual`, `LR-conf`) without deprecated penalty syntax.
   - Strict GroupKFold ($K=5$) Out-of-Fold (OOF) risk calibration on Dev set ($N=323$, 88 clusters) with zero leakage assertion.
   - Evaluation of selective risk at exact matched coverages (50%, 60%, 70%, 80%, 90%).
   - Tier-2 Reliability Metrics: ECE (15 bins), Brier score, Confidently-wrong rate (@0.90), and Odds Ratio ($D_{\text{lang}}$ vs error) with cluster-robust SE.

In [ ]:
# ==============================================================================
# STAGE 1: PURE CPU ENVIRONMENT & SYSTEM AUDIT
# ==============================================================================
import os
import sys
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# Ensure project root and src/ are in sys.path
repo_root = Path.cwd()
for p in [repo_root, repo_root.parent, repo_root / "src", repo_root.parent / "src"]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src.stats_engine import (
    calculate_discrete_aurc,
    get_risk_at_coverage,
    paired_cluster_permutation_test,
    paired_cluster_bootstrap,
    apply_holm_bonferroni,
    wilson_ci,
    run_mcnemar
)
from src.metrics import (
    compute_calibration_metrics,
    compute_visual_flip_rate,
    compute_odds_ratio_dlang,
    run_oof_abstention_pipeline,
    interpolate_risk_coverage
)

print(f"🖥️  Platform: {platform.system()} {platform.release()} ({platform.machine()})")
print(f"🐍 Python: {platform.python_version()}")
print(f"📦 NumPy: {np.__version__} | SciPy: {scipy.__version__} | Pandas: {pd.__version__} | Scikit-Learn: {sklearn.__version__}")
print("✅ System Audit Complete. Pure CPU Execution Mode confirmed.")

In [ ]:
# ==============================================================================
# STAGE 2: EXPERIMENT CONFIGURATION
# ==============================================================================
from pathlib import Path

REVISION_ID = "rev05"
AUTHORITATIVE_MANIFEST_SHA256 = "fb55fd75378b1c03abcbeb9772fa27d6ab81dd9abdfca4735c03438507c19ee9"

for cand_dir in [
    Path("artifacts/experiments/decisive_validation/rev05"),
    Path("results/decisive_validation/rev05"),
    Path("main/results/decisive_validation/rev05")
]:
    if cand_dir.exists():
        RESULTS_ROOT = cand_dir
        break
else:
    RESULTS_ROOT = Path("results/decisive_validation/rev05")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

N_PERMUTATIONS_DEFAULT = 10000
N_BOOTSTRAP_DEFAULT = 1000
RANDOM_SEED = 42

try:
    from scripts.manifest_rev05_b64 import EMBEDDED_MANIFEST_B64
except ImportError:
    try:
        from manifest_rev05_b64 import EMBEDDED_MANIFEST_B64
    except ImportError:
        EMBEDDED_MANIFEST_B64 = ""

print(f"📁 Results Directory: {RESULTS_ROOT.resolve()}")


In [ ]:
# ==============================================================================
# STAGE 3: AUTHORITATIVE FROZEN MANIFEST ENFORCEMENT
# ==============================================================================
import zlib
import base64
import hashlib
import io
import pandas as pd

def load_and_verify_manifest():
    search_paths = [
        Path("data/manifests/manifest_rev05.csv"),
        Path("artifacts/experiments/decisive_validation/rev05/manifest_rev05.csv"),
        Path("artifacts/experiments/decisive_validation/rev05/metrics/manifest_rev05.csv"),
        Path("main/results/manifest_rev05.csv"),
        Path("main/results/decisive_validation/rev05/manifest_rev05.csv"),
        Path("results/decisive_validation/rev05/manifest_rev05.csv"),
        Path("manifest_rev05.csv"),
        Path("./manifest_rev05.csv")
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.glob("**/manifest_rev05.csv"):
            search_paths.append(p)
            
    df_manifest = None
    loaded_source = None
    for p in search_paths:
        if p.exists() and p.is_file():
            raw_bytes = p.read_bytes()
            if hashlib.sha256(raw_bytes).hexdigest() == AUTHORITATIVE_MANIFEST_SHA256:
                df_manifest = pd.read_csv(io.BytesIO(raw_bytes))
                loaded_source = str(p)
                break
                
    if df_manifest is None:
        if not EMBEDDED_MANIFEST_B64:
            raise RuntimeError(f"❌ Authoritative manifest with SHA-256 {AUTHORITATIVE_MANIFEST_SHA256} not found in search paths and embedded fallback unavailable!")
        print("ℹ️ Local manifest file not found. Decompressing embedded base64 verified manifest...")
        try:
            decompressed_bytes = zlib.decompress(base64.b64decode(EMBEDDED_MANIFEST_B64))
            calc_sha = hashlib.sha256(decompressed_bytes).hexdigest()
            if calc_sha != AUTHORITATIVE_MANIFEST_SHA256:
                raise RuntimeError(f"❌ Embedded manifest hash mismatch! Expected {AUTHORITATIVE_MANIFEST_SHA256}, got {calc_sha}")
            df_manifest = pd.read_csv(io.BytesIO(decompressed_bytes))
            loaded_source = "EMBEDDED_VERIFIED_BASE64"
        except Exception as e:
            raise RuntimeError(f"❌ Failed to decompress/verify embedded manifest: {e}")
        
    if len(df_manifest) != 2244:
        raise RuntimeError(f"❌ Manifest row count mismatch! Expected 2244 total QA rows, got {len(df_manifest)}")
    print(f"✅ Authoritative Manifest 100% Cryptographically Verified ({loaded_source})")
    return df_manifest

df_manifest = load_and_verify_manifest()


In [ ]:
# ==============================================================================
# STAGE 3: DUAL-MODEL CHECKPOINT DISCOVERY & STRICT INTEGRITY AUDIT
# ==============================================================================
MODEL_1_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_2_NAME = "FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL"

def discover_checkpoint(model_name):
    slug = model_name.replace("/", "_").replace("-", "_")
    target_name = f"live_validation_checkpoint_{slug}.csv"
    
    candidates = [
        Path("artifacts/experiments/decisive_validation/rev06/predictions") / ("qwen" if "qwen" in slug.lower() and "huatuo" not in slug.lower() else "huatuo") / target_name,
        RESULTS_ROOT / slug / target_name,
        RESULTS_ROOT / target_name,
        Path("artifacts/experiments/decisive_validation/rev05/predictions") / ("qwen" if "qwen" in slug.lower() and "huatuo" not in slug.lower() else "huatuo") / target_name,
        Path("results/decisive_validation/rev05") / slug / target_name,
        Path("/kaggle/working") / slug / target_name,
        Path("/kaggle/working") / target_name,
        Path("main/results/decisive_validation/rev05") / slug / target_name
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        short_kw = "qwen" if ("qwen" in slug.lower() and "huatuo" not in slug.lower()) else "huatuo"
        for p in kaggle_input.rglob("*.csv"):
            nl = p.name.lower()
            if (short_kw in nl and "checkpoint" in nl) or slug.lower() in nl:
                candidates.insert(0, p)
                
    for p in candidates:
        if p.exists() and p.is_file() and p.stat().st_size > 1000:
            # Strictly reject mock checkpoints
            if "mock" in p.name.lower() or "mock" in str(p).lower():
                raise RuntimeError(f"❌ Mock checkpoint file rejected: {p}")
            df = pd.read_csv(p)
            if "is_mock" in df.columns and (df["is_mock"] == True).any():
                raise RuntimeError(f"❌ Mock checkpoint detected via is_mock column in {p}! Refusing mock data.")
            if (len(df) == 1133 and 
                df["revision_id"].iloc[0] in [REVISION_ID, "rev05b", "rev06"] and
                df["manifest_sha256"].iloc[0] == AUTHORITATIVE_MANIFEST_SHA256 and
                df["model_id"].iloc[0] == model_name and
                np.isfinite(df["score_m_norm_seq"]).all() and
                df["qa_id"].duplicated().sum() == 0):
                print(f"✅ Valid 100% complete checkpoint found for {model_name.split('/')[-1]}: {p}")
                return df
                
    raise FileNotFoundError(f"❌ Complete 1,133-row checkpoint for {model_name} not found!")

df_qwen = discover_checkpoint(MODEL_1_NAME)
df_huatuo = discover_checkpoint(MODEL_2_NAME)
print(f"📊 Qwen Checkpoint: {len(df_qwen)} rows, {len(df_qwen.columns)} columns")
print(f"📊 Huatuo Checkpoint: {len(df_huatuo)} rows, {len(df_huatuo.columns)} columns")


In [ ]:
# ==============================================================================
# STAGE 4: STATISTICAL EVALUATION ENGINE (IMPORT FROM SRC/STATS_ENGINE)
# ==============================================================================
# Authoritative statistical functions are imported from src.stats_engine
try:
    from src.stats_engine import (
        calculate_discrete_aurc,
        get_risk_at_coverage,
        paired_cluster_permutation_test,
        paired_cluster_bootstrap,
        apply_holm_bonferroni,
        wilson_ci,
        run_mcnemar
    )
except ImportError:
    from stats_engine import (
        calculate_discrete_aurc,
        get_risk_at_coverage,
        paired_cluster_permutation_test,
        paired_cluster_bootstrap,
        apply_holm_bonferroni,
        wilson_ci,
        run_mcnemar
    )

print("✅ Statistical Testing Engine (src.stats_engine) successfully connected.")


In [ ]:
# ==============================================================================
# STAGE 4B: AUTOMATED PREFLIGHT VERIFICATION & SANITY TESTS
# ==============================================================================
print("Running preflight mathematical invariants & index sanity checks...")

# 1. Mathematical boundary check
y_test_mock = np.array([1, 1, 0, 0])
s_test_mock = np.array([0.9, 0.8, 0.2, 0.1])
aurc_val, cov_val, risk_val = calculate_discrete_aurc(y_test_mock, s_test_mock)
assert 0.0 <= aurc_val <= 1.0, f"AURC out of bounds: {aurc_val}"

# 2. Holm-Bonferroni monotonicity check
raw_p_test = {"a": 0.04, "b": 0.01, "c": 0.02}
adj_test = apply_holm_bonferroni(raw_p_test)
assert adj_test["b"] <= adj_test["c"] <= adj_test["a"], "Holm-Bonferroni monotonicity violated!"

# 3. Tie-handling verification
y_tied = np.array([1, 0, 1, 0])
s_tied = np.array([0.5, 0.5, 0.5, 0.5])
a_best, _, _ = calculate_discrete_aurc(y_tied, s_tied, tie_mode="best")
a_worst, _, _ = calculate_discrete_aurc(y_tied, s_tied, tie_mode="worst")
a_rand, _, _ = calculate_discrete_aurc(y_tied, s_tied, tie_mode="random", n_tie_reps=50)
assert a_best < a_worst, "Tie best must be < worst"
assert a_best <= a_rand <= a_worst, "Tie random must be bounded by best and worst"

print("✅ Preflight Invariants Verification: 100% PASS.")

In [ ]:
# ==============================================================================
# STAGE 5: RQ1 BILINGUAL ACCURACY & CROSS-LINGUAL DISAGREEMENT
# ==============================================================================

def evaluate_rq1_comprehensive(df, model_title):
    subsets = [
        ("primary_test", df[(df["split"] == "test") & (df["is_eligible_primary"] == True)]),
        ("primary_dev", df[(df["split"] == "dev") & (df["is_eligible_primary"] == True)]),
        ("historical_test", df[(df["split"] == "test") & (df["in_rev03_benchmark"] == True)]),
        ("historical_dev", df[(df["split"] == "dev") & (df["in_rev03_benchmark"] == True)])
    ]
    rows = []
    for name, sub in subsets:
        n = len(sub)
        # Sequence track
        acc_en_seq = sub["correct_en_seq"].mean() * 100.0
        acc_id_seq = sub["correct_id_seq"].mean() * 100.0
        gap_seq = acc_en_seq - acc_id_seq
        flip_seq = sub["flip_seq"].mean() * 100.0
        joint_seq = sub["joint_correct_seq"].mean() * 100.0
        
        # Length-normalized track
        acc_en_len = sub["correct_en_len"].mean() * 100.0
        acc_id_len = sub["correct_id_len"].mean() * 100.0
        gap_len = acc_en_len - acc_id_len
        
        # McNemar test
        mcn_seq = run_mcnemar(sub["correct_en_seq"].values, sub["correct_id_seq"].values)
        
        # Wilson CI for primary test
        ci_en = wilson_ci(int(sub["correct_en_seq"].sum()), n)
        ci_id = wilson_ci(int(sub["correct_id_seq"].sum()), n)
        
        row_data = {
            "model": model_title, "cohort": name, "n_samples": n,
            "acc_en_seq": acc_en_seq, "acc_en_ci": (ci_en[0]*100, ci_en[1]*100),
            "acc_id_seq": acc_id_seq, "acc_id_ci": (ci_id[0]*100, ci_id[1]*100),
            "gap_seq": gap_seq,
            "flip_rate_seq": flip_seq, "joint_acc_seq": joint_seq,
            "mcnemar_p": mcn_seq["p_value_exact"],
            "acc_en_len": acc_en_len, "acc_id_len": acc_id_len, "gap_len": gap_len
        }
        
        # Include rev06 track if available
        if "correct_en_v6" in sub.columns:
            row_data.update({
                "acc_en_v6": sub["correct_en_v6"].mean() * 100.0,
                "acc_id_v6": sub["correct_id_v6"].mean() * 100.0,
                "gap_v6": (sub["correct_en_v6"].mean() - sub["correct_id_v6"].mean()) * 100.0,
                "flip_rate_v6": (sub["pred_en_v6"] != sub["pred_id_v6"]).mean() * 100.0
            })
        rows.append(row_data)
    return rows

rq1_data = evaluate_rq1_comprehensive(df_qwen, "Qwen2.5-VL-7B") + evaluate_rq1_comprehensive(df_huatuo, "HuatuoGPT-Vision-7B")
df_rq1 = pd.DataFrame(rq1_data)
rq1_csv = RESULTS_ROOT / "rq1_bilingual_metrics.csv"
df_rq1.to_csv(rq1_csv, index=False)

print("=" * 95)
print("📊 RQ1 BILINGUAL METRICS SUMMARY (PRIMARY TEST N=799)")
print("=" * 95)
cols_show = ["model", "cohort", "acc_en_seq", "acc_id_seq", "gap_seq", "flip_rate_seq", "mcnemar_p"]
display(df_rq1[df_rq1["cohort"] == "primary_test"][cols_show])
print("=" * 95)

In [ ]:
# ==============================================================================
# STAGE 6: RQ2 MULTIMODAL VISUAL INTERVENTION & VISUAL FLIP RATE
# ==============================================================================

def evaluate_rq2_comprehensive(df, model_title):
    subsets = [
        ("primary_test", df[(df["split"] == "test") & (df["is_eligible_primary"] == True)]),
        ("historical_test", df[(df["split"] == "test") & (df["in_rev03_benchmark"] == True)])
    ]
    rows = []
    for name, sub in subsets:
        n = len(sub)
        dp_seq = sub["delta_p_dir_seq"].values
        mean_dp_seq = float(np.mean(dp_seq))
        pos_seq = float(np.mean(dp_seq > 0)) * 100.0
        neg_seq = float(np.mean(dp_seq < 0)) * 100.0
        dp_len = sub["delta_p_dir_len"].values
        mean_dp_len = float(np.mean(dp_len))
        
        # Visual flip rate if pred_blank available
        vis_flip = None
        if "pred_blank_id_v6" in sub.columns:
            vis_flip = compute_visual_flip_rate(sub["pred_id_v6"].values, sub["pred_blank_id_v6"].values)["visual_flip_rate"]
            
        rows.append({
            "model": model_title, "cohort": name, "n_samples": n,
            "mean_delta_p_seq": mean_dp_seq, "prop_pos_reliance_seq": pos_seq, "prop_neg_reliance_seq": neg_seq,
            "mean_delta_p_len": mean_dp_len,
            "visual_flip_rate_p0_p1": vis_flip
        })
    return rows

rq2_data = evaluate_rq2_comprehensive(df_qwen, "Qwen2.5-VL-7B") + evaluate_rq2_comprehensive(df_huatuo, "HuatuoGPT-Vision-7B")
df_rq2 = pd.DataFrame(rq2_data)
rq2_csv = RESULTS_ROOT / "rq2_visual_intervention_metrics.csv"
df_rq2.to_csv(rq2_csv, index=False)

print("=" * 95)
print("📊 RQ2 VISUAL INTERVENTION SUMMARY (PRIMARY TEST N=799)")
print("=" * 95)
display(df_rq2[df_rq2["cohort"] == "primary_test"])
print("=" * 95)

In [ ]:
import matplotlib.pyplot as plt
# ==============================================================================
# STAGE 7: RQ3 SELECTIVE PREDICTION & RIGOROUS STATISTICAL TESTING
# ==============================================================================
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

def run_selective_prediction(df_subset, model_title, track_label="primary_test", n_perm=1000, tie_mode="random"):
    df_subset = df_subset.reset_index(drop=True)
    n_samples = len(df_subset)
    n_clusters = df_subset["canonical_image_id"].nunique()
    
    tracks_to_run = [("sequence_logprob (rev05)", "seq")]
    if "score_m1_len" in df_subset.columns:
        tracks_to_run.append(("length_normalized (rev05)", "len"))
    if "score_m1_v6" in df_subset.columns:
        tracks_to_run.append(("multi_token_logsumexp (rev06)", "v6"))
        
    all_rows = []
    
    for track_name, suffix in tracks_to_run:
        corr_col = f"correct_id_{suffix}"
        score_cols = [
            f"score_m1_{suffix}", f"score_m2_{suffix}", f"score_m_dir_{suffix}",
            f"score_m_norm_{suffix}", f"score_m3_old_{suffix}"
        ]
        
        boot_sum, boot_diff = paired_cluster_bootstrap(df_subset, score_cols, corr_col=corr_col, tie_mode=tie_mode)
        
        raw_p = {}
        curves = {}
        base_aurc = None
        base_col = f"score_m1_{suffix}"
        
        for col in score_cols:
            aurc, cov, risk = calculate_discrete_aurc(df_subset[corr_col].values, df_subset[col].values, tie_mode=tie_mode)
            curves[col] = (cov, risk)
            if col == base_col:
                base_aurc = aurc
                p_val = 1.0
                delta_aurc = 0.0
            else:
                p_val = paired_cluster_permutation_test(
                    df_subset, col, base_col, corr_col=corr_col, 
                    n_perm=n_perm, tie_mode=tie_mode, seed=42
                )
                raw_p[col] = p_val
                delta_aurc = aurc - base_aurc
                
            r100 = get_risk_at_coverage(cov, risk, 1.0)
            r90 = get_risk_at_coverage(cov, risk, 0.9)
            r80 = get_risk_at_coverage(cov, risk, 0.8)
            r70 = get_risk_at_coverage(cov, risk, 0.7)
            r50 = get_risk_at_coverage(cov, risk, 0.5)
            
            clean_est = col.replace(f"_{suffix}", "")
            all_rows.append({
                "model": model_title, "cohort": track_label, "scoring_track": track_name,
                "revision": "rev06" if suffix == "v6" else "rev05",
                "tie_mode": tie_mode,
                "n_samples": n_samples, "n_clusters": n_clusters, "estimator": clean_est,
                "aurc_mean": aurc, "ci_95_lower": boot_sum[col]["ci_lower"], "ci_95_upper": boot_sum[col]["ci_upper"],
                "delta_aurc": delta_aurc,
                "delta_ci_lower": boot_diff[col]["delta_ci_lower"] if col != base_col else 0.0,
                "delta_ci_upper": boot_diff[col]["delta_ci_upper"] if col != base_col else 0.0,
                "perm_p_raw": p_val,
                "risk_100": r100, "risk_90": r90, "risk_80": r80, "risk_70": r70, "risk_50": r50
            })
            
        adj_p = apply_holm_bonferroni(raw_p)
        for r in all_rows:
            if r["scoring_track"] == track_name:
                orig_col = f"{r['estimator']}_{suffix}"
                r["perm_p_holm"] = adj_p.get(orig_col, 1.0)
                
        if suffix in ["seq", "v6"]:
            safe_m = model_title.replace("/", "_").replace("-", "_")
            plt.figure(figsize=(8, 6), dpi=150)
            for col, (cov, risk) in curves.items():
                clean_name = col.replace(f"_{suffix}", "")
                aurc_val = [r["aurc_mean"] for r in all_rows if r["estimator"] == clean_name and r["scoring_track"] == track_name][0]
                plt.plot(cov, risk, label=f"{clean_name} (AURC={aurc_val:.4f})", lw=2)
            plt.xlabel("Coverage (Proportion of Evaluated Cases)", fontsize=11)
            plt.ylabel("Selective Diagnostic Risk (1 - Accuracy)", fontsize=11)
            plt.title(f"Risk-Coverage: {model_title} [{track_name}]\n({track_label.upper()} N={n_samples})", fontsize=12, fontweight="bold")
            plt.legend(frameon=True, fontsize=10)
            plt.grid(True, linestyle="--", alpha=0.6)
            plt.tight_layout()
            plt.savefig(RESULTS_ROOT / f"risk_coverage_{track_label}_{safe_m}_{suffix}.png")
            plt.close()
            
    return pd.DataFrame(all_rows)

df_qwen_test = df_qwen[(df_qwen["split"] == "test") & (df_qwen["is_eligible_primary"] == True)].reset_index(drop=True)
df_huatuo_test = df_huatuo[(df_huatuo["split"] == "test") & (df_huatuo["is_eligible_primary"] == True)].reset_index(drop=True)

rq3_qwen = run_selective_prediction(df_qwen_test, "Qwen2.5-VL-7B", "primary_test", n_perm=1000)
rq3_huatuo = run_selective_prediction(df_huatuo_test, "HuatuoGPT-Vision-7B", "primary_test", n_perm=1000)

df_rq3 = pd.concat([rq3_qwen, rq3_huatuo], ignore_index=True)
rq3_csv = RESULTS_ROOT / "rq3_selective_prediction_primary_test.csv"
df_rq3.to_csv(rq3_csv, index=False)

print("=" * 105)
print("🏆 RQ3 SELECTIVE PREDICTION (PRIMARY TEST N=799, 204 CLUSTERS) — SIDE-BY-SIDE rev05 vs rev06")
print("=" * 105)
cols_show = ["model", "revision", "scoring_track", "estimator", "aurc_mean", "ci_95_lower", "ci_95_upper", "delta_aurc", "perm_p_holm", "risk_100", "risk_70", "risk_50"]
display(df_rq3[cols_show])
print("=" * 105)


In [ ]:
# ==============================================================================
# STAGE 8: LEARNED RISK ABLATION, OOF ABSTENTION & RELIABILITY METRICS
# ==============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

def evaluate_risk_ablation_advanced(df_full, model_title):
    df_dev = df_full[(df_full["split"] == "dev") & (df_full["is_eligible_primary"] == True)].copy().reset_index(drop=True)
    df_test = df_full[(df_full["split"] == "test") & (df_full["is_eligible_primary"] == True)].copy().reset_index(drop=True)
    
    suffixes_to_eval = [("rev05 (sequence)", "seq")]
    if "score_m1_v6" in df_full.columns:
        suffixes_to_eval.append(("rev06 (multi-token v6)", "v6"))
        
    all_ablation_rows = []
    
    for rev_label, sfx in suffixes_to_eval:
        for df in [df_dev, df_test]:
            df[f"d_lang_{sfx}"] = (df[f"pred_en_{sfx}"] != df[f"pred_id_{sfx}"]).astype(int)
            df[f"error_id_{sfx}"] = (1 - df[f"correct_id_{sfx}"]).astype(int)
            
        # 1. Tier-2 Reliability Metrics
        calib_metrics = compute_calibration_metrics(df_test[f"conf_id_{sfx}"].values, df_test[f"correct_id_{sfx}"].values)
        or_metrics = compute_odds_ratio_dlang(df_test[f"d_lang_{sfx}"].values, df_test[f"error_id_{sfx}"].values, clusters=df_test["canonical_image_id"].values)
        print(f"\n=== {model_title} [{rev_label}] Reliability Metrics (Primary Test N={len(df_test)}) ===")
        print(f"  ECE (15 bins): {calib_metrics['ece']:.4f} | Brier: {calib_metrics['brier']:.4f}")
        print(f"  Confidently-Wrong Rate (@0.90): {calib_metrics['conf_wrong_rate']*100:.2f}%")
        print(f"  Odds Ratio D_lang: {or_metrics['odds_ratio']:.4f} (p={or_metrics['p_value']:.4e})")
        if or_metrics.get("cluster_robust_se") is not None:
            print(f"  Cluster-Robust Logit Coef: {or_metrics['logit_coef']:.4f} ± {or_metrics['cluster_robust_se']:.4f} (p={or_metrics['logit_pvalue']:.4e})")
            
        # 2. Risk Ablation Models
        feature_sets = {
            "LR_conf": [f"conf_id_{sfx}"],
            "LR_visual": [f"conf_id_{sfx}", f"delta_p_dir_{sfx}"],
            "LR_full": [f"conf_id_{sfx}", f"delta_p_dir_{sfx}", f"d_lang_{sfx}"]
        }
        
        y_dev = df_dev[f"error_id_{sfx}"].values
        y_test = df_test[f"error_id_{sfx}"].values
        
        preds_test = {}
        for m_name, f_cols in feature_sets.items():
            scaler = StandardScaler()
            X_dev = scaler.fit_transform(df_dev[f_cols].values)
            X_test = scaler.transform(df_test[f_cols].values)
            
            clf = LogisticRegression(C=1.0, random_state=42)
            clf.fit(X_dev, y_dev)
            p_risk = clf.predict_proba(X_test)[:, 1]
            s_score = 1.0 - p_risk
            preds_test[m_name] = s_score
            
            aurc_val, cov_arr, risk_arr = calculate_discrete_aurc(df_test[f"correct_id_{sfx}"].values, s_score, tie_mode="random")
            rc_interp = interpolate_risk_coverage(cov_arr, risk_arr, [0.5, 0.6, 0.7, 0.8, 0.9])
            
            all_ablation_rows.append({
                "model": model_title, "revision": rev_label, "feature_set": m_name,
                "aurc": aurc_val,
                "risk_90": rc_interp.get(0.9, np.nan),
                "risk_80": rc_interp.get(0.8, np.nan),
                "risk_70": rc_interp.get(0.7, np.nan),
                "risk_60": rc_interp.get(0.6, np.nan),
                "risk_50": rc_interp.get(0.5, np.nan)
            })
            
        # 3. OOF Abstention Pipeline
        oof_res = run_oof_abstention_pipeline(
            df_dev, df_test,
            feature_cols=feature_sets["LR_full"],
            error_col=f"error_id_{sfx}",
            cluster_col="canonical_image_id",
            target_coverages=[0.9, 0.8, 0.7, 0.5]
        )
        print(f"  OOF Thresholds calibrated on Dev (K=5): {oof_res['oof_thresholds']}")
        print(f"  Test Evaluation at OOF-Calibrated Thresholds:")
        for _, r_oof in oof_res['test_results'].iterrows():
            print(f"    Target Cov: {r_oof['target_coverage']*100:.0f}% -> Realized Cov: {r_oof['realized_coverage']:.2f}% | Sel Risk: {r_oof['selective_risk']:.2f}% (Acc: {r_oof['selective_accuracy']:.2f}%)")

    return pd.DataFrame(all_ablation_rows)

abl_qwen = evaluate_risk_ablation_advanced(df_qwen, "Qwen2.5-VL-7B")
abl_huatuo = evaluate_risk_ablation_advanced(df_huatuo, "HuatuoGPT-Vision-7B")

df_ablation = pd.concat([abl_qwen, abl_huatuo], ignore_index=True)
abl_csv = RESULTS_ROOT / "risk_ablation_dev_test.csv"
df_ablation.to_csv(abl_csv, index=False)

print("\n" + "=" * 95)
print("📊 LEARNED RISK ABLATION & EXACT COVERAGE-RISK INTERPOLATION (PRIMARY TEST)")
print("=" * 95)
display(df_ablation)
print("=" * 95)


In [ ]:
# ==============================================================================
# STAGE 9: ARTIFACT SHA-256 MANIFEST & FINAL REPRODUCIBILITY ARCHIVE
# ==============================================================================
import json
import zipfile
import shutil
import hashlib
from IPython.display import FileLink
from datetime import datetime, timezone

# 1. Generate run_metadata.json
metadata = {
    "revision_id": REVISION_ID,
    "manifest_sha256": AUTHORITATIVE_MANIFEST_SHA256,
    "completed_timestamp": datetime.now(timezone.utc).isoformat(),
    "qwen_completed_samples": len(df_qwen),
    "huatuo_completed_samples": len(df_huatuo),
    "primary_test_samples": 799,
    "primary_dev_samples": 323,
    "historical_test_samples": 92,
    "historical_dev_samples": 58
}
meta_p = RESULTS_ROOT / "run_metadata.json"
meta_p.write_text(json.dumps(metadata, indent=2))

# 2. Artifact SHA-256 Manifest
artifact_rows = []
for root, dirs, files in os.walk(RESULTS_ROOT):
    for f in files:
        if f.endswith(".zip") or f.endswith(".tmp"):
            continue
        fp = Path(root) / f
        h = hashlib.sha256(fp.read_bytes()).hexdigest()
        artifact_rows.append({"file_name": fp.name, "relative_path": str(fp.relative_to(RESULTS_ROOT)), "sha256": h})

df_art = pd.DataFrame(artifact_rows)
art_csv = RESULTS_ROOT / "artifact_sha256_manifest.csv"
df_art.to_csv(art_csv, index=False)

# 3. Create Final Reproducibility Zip
final_zip_name = "decisive_validation_rev05_complete.zip"
final_zip_path = Path("/kaggle/working") / final_zip_name if Path("/kaggle/working").exists() else RESULTS_ROOT / final_zip_name

with zipfile.ZipFile(final_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(RESULTS_ROOT):
        for f in files:
            p = Path(root) / f
            if p != final_zip_path and not p.name.endswith(".tmp"):
                zf.write(p, arcname=p.relative_to(RESULTS_ROOT))

print("=" * 80)
print(f"📦 REPRODUCIBILITY ARCHIVE COMPLETE: {final_zip_path.name}")
print(f"   Size: {final_zip_path.stat().st_size / (1024*1024):.2f} MB")
print("=" * 80)
display(FileLink(str(final_zip_path)))